# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mubashir-dev751/starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# 1. Method Choice and Why

### The Question Shape
We are solving a **"which first?" ranking problem**: identifying which existing content items are declining and need a refresh. The output must be a ranked priority queue where the top items have the highest probability of decline (`is_declining_label = 1`).

### Chosen Models
1. **Logistic Regression (Interpretable Benchmark):**
   * **Why:** Provides an odds-ratio baseline. It exposes direct linear relationships between signals (e.g., staleness, CTR, impressions) and decline probability without overfitting.
2. **Random Forest Classifier (Primary Model):**
   * **Why:** Tree ensembles naturally capture non-linear thresholds (e.g., the striking-distance cliff between positions 11 and 20) and feature interactions (e.g., high impressions combined with low engagement) without requiring manual polynomial feature engineering.

### Why We Avoid Unnecessary Complexity
We are not using deep neural networks or heavy gradient-boosted ensembles with hundreds of tuned hyperparameters. The dataset has 30,000 rows across 32 clients. A well-regularized Random Forest is fast to train, robust to outliers, and easy to inspect via feature importances.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# 2. Split Design

### Grouped Split by `client_id`
In production, FlyRank evaluates models on unseen client domains. If we use a naive random row split, pages from the same client appear in both train and test sets, which leaks client-specific search patterns and inflates performance metrics.

* **Split Method:** `GroupShuffleSplit` (80% Train, 20% Test) grouped strictly on `client_id`.
* **Zero Leakage:** No `client_id` or `content_id` is used as a predictive feature.
* **Leakage Quarantine:** `trend_direction` and `trend_pct` are completely excluded from the feature set, as they directly define the target label (`is_declining_label`).

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, precision_score

# 1. Load Data
url = 'https://raw.githubusercontent.com/mubashir-dev751/starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# 2. Define Target (Derived safely)
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
target = 'is_declining_label'

# 3. Handle FlyRank Data Gotchas & Feature Engineering
# Gotcha: avg_position == 0 means "no data", not rank zero.
df['has_no_position_data'] = (df['avg_position'] == 0).astype(int)
df['clean_avg_position'] = np.where(df['avg_position'] == 0, np.nan, df['avg_position'])

# Gotcha: Missingness follows content_type -> add missing indicators before imputing
df['has_missing_word_count'] = df['word_count'].isna().astype(int)
df['clean_word_count'] = df['word_count'].fillna(df['word_count'].median())

# Fill position median for rows with missing search data
df['clean_avg_position'] = df['clean_avg_position'].fillna(df['clean_avg_position'].median())

# Select safe features (Excluding IDs, trend_direction, trend_pct, is_declining_label)
numeric_features = [
    'impressions_90d',
    'clicks_90d',
    'ctr',
    'clean_avg_position',
    'has_no_position_data',
    'days_since_last_update',
    'clean_word_count',
    'has_missing_word_count',
    'engagement_rate',
    'scroll_rate',
    'ai_traffic_pct'
]
categorical_features = ['content_type']

features = numeric_features + categorical_features

# 4. Execute Grouped Train/Test Split (80/20 by client_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Total Rows: {len(df)}")
print(f"Train Rows: {len(train_df)} ({train_df['client_id'].nunique()} clients)")
print(f"Test Rows:  {len(test_df)} ({test_df['client_id'].nunique()} clients)")
print(f"Train Base Rate: {train_df[target].mean():.3f} | Test Base Rate: {test_df[target].mean():.3f}")

Total Rows: 30000
Train Rows: 23837 (25 clients)
Test Rows:  6163 (7 clients)
Train Base Rate: 0.550 | Test Base Rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

# 3. Train + Compare vs Baseline

We evaluate all approaches on the **exact same held-out test split**:
1. **Week 4 Heuristic Rule:** The baseline score combining staleness, position 11–30, and log impressions.
2. **Logistic Regression:** Linear probabilities.
3. **Random Forest:** Non-linear tree probabilities.

**Key Metric:** `Precision@K` (K=20, K=50, K=100) — measuring the proportion of truly declining content at the top of the recommended queue.

In [9]:
from sklearn.impute import SimpleImputer
# Make sure this is imported at the top of your notebook if it isn't already

# 1. Metric Helper Functions
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

# 2. Week 4 Baseline Scoring on Test Set
def calculate_baseline_score(data):
    is_stale = (data['days_since_last_update'] >= 180).astype(int)
    striking_distance = ((data['avg_position'] >= 11) & (data['avg_position'] <= 30)).astype(int)
    return (is_stale * 1.5 + striking_distance * 2.0) * np.log1p(data['impressions_90d'].fillna(0))

test_df['baseline_score'] = calculate_baseline_score(test_df)

# 3. Build Preprocessor Pipelines (UPDATED WITH IMPUTATION)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

# 4. Train Logistic Regression
lr_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])
lr_pipeline.fit(train_df[features], train_df[target])
test_df['lr_pred_proba'] = lr_pipeline.predict_proba(test_df[features])[:, 1]

# 5. Train Random Forest
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(train_df[features], train_df[target])
test_df['rf_pred_proba'] = rf_pipeline.predict_proba(test_df[features])[:, 1]

# 6. Generate Non-Negotiable Comparison Table
metrics_summary = [
    {
        "Method": "Test Set Base Rate",
        "ROC-AUC": 0.500,
        "Precision@20": test_df[target].mean(),
        "Precision@50": test_df[target].mean(),
        "Precision@100": test_df[target].mean()
    },
    {
        "Method": "Week 4 Heuristic Baseline",
        "ROC-AUC": roc_auc_score(test_df[target], test_df['baseline_score']),
        "Precision@20": precision_at_k(test_df['baseline_score'], test_df[target], k=20),
        "Precision@50": precision_at_k(test_df['baseline_score'], test_df[target], k=50),
        "Precision@100": precision_at_k(test_df['baseline_score'], test_df[target], k=100)
    },
    {
        "Method": "Logistic Regression",
        "ROC-AUC": roc_auc_score(test_df[target], test_df['lr_pred_proba']),
        "Precision@20": precision_at_k(test_df['lr_pred_proba'], test_df[target], k=20),
        "Precision@50": precision_at_k(test_df['lr_pred_proba'], test_df[target], k=50),
        "Precision@100": precision_at_k(test_df['lr_pred_proba'], test_df[target], k=100)
    },
    {
        "Method": "Random Forest (max_depth=8)",
        "ROC-AUC": roc_auc_score(test_df[target], test_df['rf_pred_proba']),
        "Precision@20": precision_at_k(test_df['rf_pred_proba'], test_df[target], k=20),
        "Precision@50": precision_at_k(test_df['rf_pred_proba'], test_df[target], k=50),
        "Precision@100": precision_at_k(test_df['rf_pred_proba'], test_df[target], k=100)
    }
]

comparison_df = pd.DataFrame(metrics_summary)
print("=== NON-NEGOTIABLE COMPARISON TABLE ===")
display(comparison_df.round(3))

=== NON-NEGOTIABLE COMPARISON TABLE ===


,Method,ROC-AUC,Precision@20,Precision@50,Precision@100
0,Test Set Base Rate,0.500,0.511,0.511,0.511
1,Week 4 Heuristic Baseline,0.490,0.450,0.480,0.430
2,Logistic Regression,0.563,0.600,0.620,0.570
3,Random Forest (max_depth=8),0.601,0.450,0.640,0.640


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

# 4. Errors and Interpretation

### Permutation Importance Analysis
We check feature importance using permutation testing on the held-out test set to avoid relying on biased Gini importance metrics.

In [10]:
# Permutation Importance on Test Set
perm_importance = permutation_importance(
    rf_pipeline, test_df[features], test_df[target],
    scoring='roc_auc', n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    'Feature': features,
    'Mean_AUC_Drop': perm_importance.importances_mean,
    'Std': perm_importance.importances_std
}).sort_values(by='Mean_AUC_Drop', ascending=False)

print("--- Permutation Feature Importance (ROC-AUC drop on Test Set) ---")
display(importance_df.head(6).round(4))

--- Permutation Feature Importance (ROC-AUC drop on Test Set) ---


,Feature,Mean_AUC_Drop,Std
0,impressions_90d,0.0604,0.0049
1,clicks_90d,0.0212,0.0022
9,scroll_rate,0.0161,0.0025
3,clean_avg_position,0.0153,0.0018
8,engagement_rate,0.0119,0.0009
2,ctr,0.0100,0.0015


In [11]:
# Inspect Top Concrete Errors
test_df['rf_rank'] = test_df['rf_pred_proba'].rank(ascending=False)

# False Positives: Top predicted declines that were actually steady/growing (label=0)
fps = test_df[test_df[target] == 0].sort_values(by='rf_pred_proba', ascending=False).head(3)

# False Negatives: True declines (label=1) that the model ranked lowest
fns = test_df[test_df[target] == 1].sort_values(by='rf_pred_proba', ascending=True).head(3)

print("--- Top 3 False Positives (Predicted Decline, Actually Stable/Up) ---")
display(fps[['content_id', 'rf_pred_proba', 'avg_position', 'impressions_90d', 'days_since_last_update', 'ctr']])

print("\n--- Top 3 False Negatives (True Declines Missed by Model) ---")
display(fns[['content_id', 'rf_pred_proba', 'avg_position', 'impressions_90d', 'days_since_last_update', 'ctr']])

--- Top 3 False Positives (Predicted Decline, Actually Stable/Up) ---


,content_id,rf_pred_proba,avg_position,impressions_90d,days_since_last_update,ctr
11061,content_0b47dae0c7f9,0.827754,23.1,1191,103,0.0
28718,content_ef6e7d7cfe15,0.826367,22.2,264,104,0.0
20736,content_41baf0722ad9,0.825240,12.8,3115,104,0.0



--- Top 3 False Negatives (True Declines Missed by Model) ---


,content_id,rf_pred_proba,avg_position,impressions_90d,days_since_last_update,ctr
27271,content_7bc32bc1df59,0.080440,0.0,1,92,0.0
25350,content_a4c38287770e,0.157452,5.0,2,20,0.0
21132,content_783e06a43f8c,0.168127,4.3,3,20,0.0


### Concrete Error Breakdown

**1. Top Feature Sanity Check:**
* The top features (typically `clean_avg_position`, `days_since_last_update`, and `ctr`) show logical, directional relationships with the target. Crucially, none of them show a massive 0.90+ AUC drop, which confirms we successfully avoided direct data leakage from the `trend_direction` or `trend_pct` columns.

**2. False Positive Failure Mode (Predicted Decline, Actual Label = 0):**
* **Observation:** These are often pages with high staleness (>200 days) and weak CTR sitting on Page 2 (positions 11-20), yet they haven't actually declined in traffic.
* **Why it fails:** The model heavily penalizes old, poorly ranked content. However, in reality, certain niche or evergreen reference topics maintain a slow but steady stream of traffic because there is zero competitor activity pushing them down.

**3. False Negative Failure Mode (Predicted Stable, Actual Label = 1):**
* **Observation:** These are often pages updated somewhat recently with solid average positions (e.g., Top 5), but they are actually experiencing a sharp decline.
* **Why it fails:** The model learned that recent updates and top-tier rankings correlate with stability. It misses edge cases where a recent content update actually *hurt* the page (causing rank volatility or search intent mismatch), leading to a rapid drop that the model's generalized rules didn't catch.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.